### 1. Basic Tasks

In [0]:
%sql
-- define schema
create schema if not exists cyntexa_dev.bronze;
create schema if not exists cyntexa_dev.silver;
create schema if not exists cyntexa_dev.gold;

  1. Create a bronze table that ingests raw sales data as-is (no transformations), preserving all original columns plus an ingestion timestamp.


Ingested raw csv file into bronze schema using spark read and write APIs as-is (no transformations) in ingestion.py file

2. Build a silver table from bronze that removes duplicates, fixes data types, and drops clearly invalid rows.

Cleaned data by handling nulls and duplicates and fixing data types, stored cleaned data table in silver schema in silver.py file

3. Build a gold table that aggregates silver into a business-ready view (e.g., daily revenue by store).

Build gold table aggregation (revenue by each customer) in gold.sql

### 2. Intermediate Tasks

### 4. 
![image_1787727950356.png](./image_1787727950356.png "image_1787727950356.png")

- Bronze: 
    - Primary consumers: Data Engineers
    - Contains raw ingested data without and transformations or cleaning .
- Silver:
    - Primary consumers: Data Analysts
    - Bronze tables are transformed into cleaned data by handling nulls, duplicates, data type etc. adn stored to silver
- Gold:
    - Primary consumers: Executives
    - Conatins business ready tables, aggregations and KPIs, used for reporting and dashboard.

### 5 
The silver transformation was recreated using lakeflow designer interface. The visual approach was faster for simple transformations such as filtering, selection columns and handling missing values. It is more user freindly to undestand visually for users with less coding experience. But writing tranformation code gives more flexiblit and control for complex tranformations and logics 

### 6. 
![image_1787727950356.png](./image_1787727950356.png "image_1787727950356.png")
Created a Databricks job implementing the Medallion Architecture with Bronze, silver and gold layers. The job orchestrates the data flow from raw data ingestion to cleaning data and finally to businedd ready KPIs

### 3. Advanced Tasks

In [0]:
%sql
-- 7.
create table cyntexa_dev.gold.inventory_summary as
select product_id, sum(quantity) as units_sold, sum(sale_amount) as total_sales,
count(distinct sale_id) as number_of_sales
from cyntexa_dev.silver.sales_cleaned
group by product_id
order by total_sales desc

In [0]:
%sql
select * from cyntexa_dev.gold.inventory_summary


Extended the Gold layer with inventory summary containing
total units sold, total sales and number of sales for each product. This helps
the inventory team identify product demand and support inventory planning.

This belongs in the Gold layer because it is a reusable, business-ready
aggregation that can be consistently used by dashboards and other consumers,
rather than being repeatedly calculated ad hoc by the inventory team.

#### 8.
The data processing and transformations should run in the customer's plane. This include reading data, running Spark transformations, cleaning, joins, aggregations and writing the output tables. This keeps the customer's data and processing within customer's cloud environment

The Databricks control plane is mainly responsible for managing the Databricks
service, such as the workspace interface, job scheduling, configuration and
orchestration metadata. The control plane should not be responsible for the
main data processing.

During a security review, the organization should ensure that communication
between the control plane and data plane is securely configured. Network access,
firewalls, private connectivity, identity and access controls, encryption and
Unity Catalog permissions should be reviewed.

In [0]:
# 9.
from pyspark.sql import functions as F

df = spark.read.table("cyntexa_dev.silver.sales_cleaned")
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()